## 0. Load data

In [29]:
#Change this variable to True when you want to use the training data, change to False when you want to use the testing data. This will allow you to easily switch between the two datasets without having to change the file paths in multiple places in the code.
IsTraining = True

In [30]:
import sys
sys.path.insert(0, '..')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
from pathlib import Path

if IsTraining:
    churn_path = Path("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/datasets/resort_train.csv")
else:
    churn_path = Path("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/datasets/resort_test.csv")

churn = pd.read_csv(churn_path)

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 80)

## C. Fill in the Nulls in Promo codes with "NoPromoCode"

In [31]:
#In PromoCode column, replace the null values with "NoPromoCode"
churn['PromoCode'] = churn['PromoCode'].fillna('NoPromoCode')



# D. Fill in the nulls in Region

In [32]:
#In Region if null fill with "Unknown"
churn['Region'] = churn['Region'].fillna('Unknown')

# E. Fill in the null in the AllInclusive column

In [33]:
#fill in the null in the AllInclusive column with "0", assume if no data they are not all inclusive
churn['AllInclusive'] = churn['AllInclusive'].fillna(0)

# F. Room column - Split into RoomFloor, RoomNumber, and RoomSide

## Is there any information we can get from guests that shared a room?

In [34]:
#I want to know how many guests shared a room
#Create a new column called "SharedRoom" that is contains a 1 if there is another GuesID that shares the same Room and BookingDate, and 0 otherwise.
churn['SharedRoom'] = churn.duplicated(subset=['Room', 'BookingDate'], keep=False).astype(int)

#Count the number of guests that shared a room
num_shared = churn['SharedRoom'].sum()
print(f'Number of guests that shared a room: {num_shared}')

Number of guests that shared a room: 251


In [35]:
#The Room column has data that is 3 sets of data separated by a forward slash. We will split this column into three separate columns, RoomFloor, RoomNumber, and RoomSide.
churn[['RoomFloor', 'RoomNumber', 'RoomSide']] = churn['Room'].str.split('/', expand=True)

#Any nulls in the RoomFloor, RoomNumber, and RoomSide columns will be filled with "Unknown", assume if there is no data there was an error in the data entry and we cannot use that row for our analysis.
#The RoomNumber column should be an int
churn['RoomFloor'] = churn['RoomFloor'].fillna('Unknown')
churn['RoomNumber'] = churn['RoomNumber'].fillna('0').astype(int)
churn['RoomSide'] = churn['RoomSide'].fillna('Unknown')

In [36]:
#drop the Room column
#churn = churn.drop(columns=['Room'])

## B. BookingDate

In [37]:

#encode the BookingDate column to a datetime object and create new columns for the month and year of the booking
churn['BookingDate'] = pd.to_datetime(churn['BookingDate'])
churn['BookingMonth'] = churn['BookingDate'].dt.month
churn['BookingYear'] = churn['BookingDate'].dt.year

#drop the BookingDate column
churn.drop(columns=['BookingDate'], inplace=True)


# G. Fill the nulls in the PackageType column

In [38]:
#Fill the nulls in the PackageType column with "NoPackage"
churn['PackageType'] = churn['PackageType'].fillna('NoPackage')

# H&S. Remove as many nulls in the age and AgeGroup as possible

We can use information contained in other columns to extrapolate some of the null information

**Information from the AgeGroup to fill in the Age column will skew the data slightly, be careful if you want to make conclusions based off the age column. To start with we will drop this column**

In [39]:
churn["AgeGroup"].value_counts()

AgeGroup
Young      2586
Middle     1810
Minor      1372
Senior      727
Elderly     176
Name: count, dtype: int64

In [40]:
#count the number of null values in AgeGroup column and  Age column
print(churn["AgeGroup"].isnull().sum())
print(churn["Age"].isnull().sum())

283
476


### Data Cleaning - Missing values - Age Group and Age

Minor      1372 1-18    10

Young      2586 19-30   24

Middle     1810 31-45   38

Senior      727 46-60   52

Elderly     176 61-79   70

we can use this data to fill in blanks in either group - I'll use the median age for the groups to fill in missing age values

In [41]:
#find null values in AgeGroup column and for each corresponding row, fill in the agegroup based on the age column. "Minor" if age >0 && <= 18, "Young" if age > 18 && <=30, "Middle" if age > 30 && <= 45, "Senior" if age > 45 && <= 60, "Elderly" if age > 60. 

mask = churn['AgeGroup'].isna() & churn['Age'].notna()

churn.loc[mask, 'AgeGroup'] = pd.cut(
    churn.loc[mask, 'Age'],
    bins=[0, 18, 30, 45, 60, float('inf')],
    labels=['Minor', 'Young', 'Middle', 'Senior', 'Elderly'],
    right=True,            # right=True means upper bound is inclusive: (lo, hi]
    include_lowest=True,  # age must be >= 0
)

print(churn["AgeGroup"].isnull().sum())
print(churn["Age"].isnull().sum())

145
476


In [42]:
#now we need to do the same thing for the Age column, find null values in Age column and for each corresponding row, fill in the age based on the agegroup column. 10 if agegroup is "Minor", 24 if agegroup is "Young", 38 if agegroup is "Middle", 52 if agegroup is "Senior", 70 if agegroup is "Elderly".
mask = churn['Age'].isna() & churn['AgeGroup'].notna()
churn.loc[mask, 'Age'] = churn.loc[mask, 'AgeGroup'].map({
    'Minor': 10,
    'Young': 24,
    'Middle': 38,
    'Senior': 52,
    'Elderly': 70
})

print(churn["AgeGroup"].isnull().sum())
print(churn["Age"].isnull().sum())

#The displayed 2 numbers should both be the same, this means that the remaining values are null in both columns and we cannot fill them in based on the other column. We may have to drop these rows later on.

145
145


In [43]:
#We have skewed the age column so let's drop the Age column
churn = churn.drop("Age", axis=1)

In [44]:
#fill in the remaining null values in the AgeGroup column with "Unknown"
churn['AgeGroup'] = churn['AgeGroup'].fillna('Unknown')

# I. Fill the nulls in VIP column

In [45]:
#In the VIP column replace the null values with "0", assume if no data they are not VIPs
churn['VIP'] = churn['VIP'].fillna(0)

# J-N. Clean the outliers from the RoomService, Dining, Retail, Spa, and Entertainment columns

In [46]:
#in the Dining, Retail, Spa, and Entertainment columns, fill in the nulls with "0", assume if no data they did not use those amenities
amenities = ['RoomService', 'Dining', 'Retail', 'Spa', 'Entertainment']
for amenity in amenities:
    churn[amenity] = churn[amenity].fillna(0)

In [47]:
#There are outliers in the Dining, Retail, Spa, and Entertainment columns. They all have decimal values, lets take any non-integer value and convert them to 0, assume if they have a decimal value they did not use that amenity.
#for amenity in amenities:
 #   churn[amenity] = churn[amenity].apply(lambda x: 0 if isinstance(x, float) and not x.is_integer() else x)    

In [48]:
#There are outliers in the Dining, Retail, Spa, and Entertainment columns. For any value that has a decimal value, take the average of the other 4 columns and fill in the decimal value with that average, assume if they have a decimal value they used that amenity but there was an error in the data entry and it was recorded as a decimal instead of an integer. If there are more than one cell that contains decimal places in the row then use the averages from the cells that do not contain decimal places to fill in the cells that do contain decimal places. If all 5 cells contain decimal places then fill them in with 0, assume if they have a decimal value they used that amenity but there was an error in the data entry and it was recorded as a decimal instead of an integer, but if all 5 cells contain decimal places then we cannot use the averages to fill in the values so we will assume they did not use any of the amenities and fill them in with 0. Keep the averages a integer format since the original data is in integer format, if the average is a decimal then round it to the nearest integer before filling in the values.
for index, row in churn.iterrows():
    decimal_cols = [col for col in amenities if isinstance(row[col], float) and not row[col].is_integer()]
    if len(decimal_cols) > 0:
        non_decimal_cols = [col for col in amenities if col not in decimal_cols]
        if len(non_decimal_cols) > 0:
            avg = row[non_decimal_cols].mean()
            for col in decimal_cols:
                churn.at[index, col] = round(avg)
        else:
            for col in decimal_cols:
                churn.at[index, col] = 0

#standardize the amenities columns using standardization, we will use the z score Standardization from sklearn to standardize the RoomService, Dining, Retail, Spa, and Entertainment columns. This will help to improve the performance of our model since these columns have different scales and we want to bring them to the same scale. We will fit the scaler on the training data and then transform both the training and testing data using the same scaler. This will ensure that the scaling is consistent between the training and testing data. We will also convert the standardized values back to a dataframe so that we can easily concatenate it with the other features later on.
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
churn[amenities] = scaler.fit_transform(churn[amenities])

In [49]:
#There are outliers in the Dining, Retail, Spa, and Entertainment columns. For any value in each of these columns that contains a decimal value, drop the whole row, assume if they have a decimal value there was an error in the data entry and we cannot use that row for our analysis.
#for amenity in amenities:
#    churn = churn[~(churn[amenity].apply(lambda x: isinstance(x, float) and not x.is_integer()))]

# Drop columns that aren't needed anymore

In [50]:
#drop Room column
churn = churn.drop(columns=['Room'])

# Convert the remaining categorical columns

In [51]:
#encode the remaining categorical columns using one hot encoding
churn = pd.get_dummies(churn, columns=['PromoCode', 'Region', 'PackageType', 'AgeGroup', 'RoomFloor', 'RoomSide', 'BookingChannel', 'ReferralSource'], drop_first=True)

# Need to convert of all the float columns left to int

In [52]:
#Convert all columns of type float to int
for col in churn.select_dtypes(include=['float']).columns:
    churn[col] = churn[col].astype(int)

In [53]:
churn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 61 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   GuestID                     6954 non-null   int64
 1   AllInclusive                6954 non-null   int64
 2   VIP                         6954 non-null   int64
 3   RoomService                 6954 non-null   int64
 4   Dining                      6954 non-null   int64
 5   Retail                      6954 non-null   int64
 6   Spa                         6954 non-null   int64
 7   Entertainment               6954 non-null   int64
 8   LoyaltyPoints               6954 non-null   int64
 9   SurveyScore                 6954 non-null   int64
 10  DaysSinceEmail              6954 non-null   int64
 11  Churned                     6954 non-null   int64
 12  SharedRoom                  6954 non-null   int64
 13  RoomNumber                  6954 non-null   int64
 14  BookingM

## Ensure the test set has 1739 rows, the training set should have 6954 rows

In [54]:
#count the number of rows in the dataset
num_rows = churn.shape[0]
print(f'Number of rows in the dataset: {num_rows}')

Number of rows in the dataset: 6954


In [55]:
#display any null values in the dataset
null_values = churn.isnull().sum()
print("Null values in each column:")
print(null_values[null_values > 0])

Null values in each column:
Series([], dtype: int64)


## Export the cleaned data

In [56]:
#Export the churned data to a csv file
if IsTraining:
    churn.to_csv("churn_train_cleaned.csv", index=False)
else:
    churn.to_csv("churn_test_cleaned.csv", index=False)